# 第三十一课:量化——把权重压进更小的整数

讲义:[31-quantization.md](31-quantization.md)。在本地 SmolLM2-135M 的真实权重上逐张量实测 int8/int4 误差与体积账。

In [ ]:
# ===== safetensors 加载 + 量化器(讲义 §3) =====
import glob
import json
import struct
import numpy as np

SNAP = glob.glob("/Volumes/Sam-APFS/huggingface/hub/models--HuggingFaceTB--"
                 "SmolLM2-135M-Instruct/snapshots/*")[0]
ST = f"{SNAP}/model.safetensors"

def read_header(st_path):
    with open(st_path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(n))

_HDRL = None
def hdrlen(st_path):
    global _HDRL
    if _HDRL is None:
        with open(st_path, "rb") as f:
            _HDRL = struct.unpack("<Q", f.read(8))[0]
    return _HDRL

def load_tensor(st_path, name, header=None):
    """逐张量加载:safetensors 原始字节 → numpy fp32。bf16 = fp32 高 16 位。"""
    header = header or read_header(st_path)
    meta = header[name]
    beg, end = meta["data_offsets"]
    with open(st_path, "rb") as f:
        f.seek(8 + hdrlen(st_path) + beg)
        raw = f.read(end - beg)
    arr = np.frombuffer(raw, dtype={"F32": np.float32, "BF16": np.uint16,
                                    "F16": np.float16}[meta["dtype"]])
    if meta["dtype"] == "BF16":
        arr = (arr.astype(np.uint32) << 16).view(np.float32)
    return arr.reshape(meta["shape"])

def quantize_sym(x, bits):
    qmax = 2 ** (bits - 1) - 1
    scale = np.abs(x).max(axis=1, keepdims=True) / qmax + 1e-12
    q = np.clip(np.round(x / scale), -qmax - 1, qmax)
    return q, scale

def quantize_group(x, bits, group=64):
    qmax = 2 ** (bits - 1) - 1
    out, in_ = x.shape
    pad = (-in_) % group
    xp = np.concatenate([x, np.zeros((out, pad))], 1).reshape(out, -1, group)
    scale = np.abs(xp).max(axis=2, keepdims=True) / qmax + 1e-12
    q = np.clip(np.round(xp / scale), -qmax - 1, qmax)
    return (q.reshape(out, -1)[:, :in_],
            np.repeat(scale, group, axis=2).reshape(out, -1)[:, :in_])

In [ ]:
# ===== 真实张量误差表 + 体积账(讲义 §4–5) =====
header = read_header(ST)
names = [k for k in header if k != "__metadata__"]
print(f"张量总数 {len(names)}")

targets = [n for n in names if n in (
    "model.layers.0.self_attn.q_proj.weight",
    "model.layers.0.mlp.down_proj.weight",
    "model.layers.5.self_attn.o_proj.weight",
    "model.embed_tokens.weight")]
print(f"{'tensor':44s} {'shape':16s} {'int8':>10s} {'int4(g64)':>10s}")
for name in sorted(targets):
    x = load_tensor(ST, name).astype(np.float32)
    q8, s8 = quantize_sym(x, 8)
    e8 = np.abs(q8 * s8 - x).max() / (np.abs(x).max() + 1e-12)
    q4, s4 = quantize_group(x, 4)
    e4 = np.abs(q4 * s4 - x).max() / (np.abs(x).max() + 1e-12)
    print(f"{name:44s} {str(x.shape):16s} {e8:10.2e} {e4:10.2e}")

n_params = sum(int(np.prod(header[n]["shape"])) for n in names)
print(f"\n全模型参数(逐张量对账):{n_params:,}")
print(f"bf16 = {n_params*2/1024**3:.3f} GB | int8 = {n_params/1024**3:.3f} GB"
      f" | int4+g64 ≈ {(n_params*0.5 + n_params/64*4)/1024**3:.3f} GB")
print("lm_head 在文件里吗?", "lm_head.weight" in header, "(词嵌入绑定)")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): (64,128) 随机矩阵第一组 scale 手算对照

# TODO(C3): int4 误差直方图,观察台阶效应

# TODO(D1): Qwen3-4B int4 体积 + KV 账,16GB 内存并发数

# TODO(E1): group ∈ {32,64,128,256} 扫描
